# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

## Preparation 

In [2]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import anthropic
import google.generativeai
import gradio as gr

In [3]:
# constants
HEADERS = {"Content-Type": "application/json"}
MODEL_GPT = "gpt-4o-mini"
MODEL_CLAUDE ="claude-sonnet-4-20250514"
GEMINI_MODEL = "gemini-2.5-flash"
DEEPSEEK_MODEL = "deepseek-chat"
MODEL_LLAMA = 'llama3.2'

In [4]:
# set up environment
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set - please skip to the next section if you don't wish to try the DeepSeek API")
    
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AIzaSyBy
DeepSeek API Key exists and begins sk-


In [5]:
# Initiate providers
openai = OpenAI()
claude = anthropic.Anthropic()
google.generativeai.configure()
deepseek_via_openai_client = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com")
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [6]:
# system prompt defintion
system_message = "You are a very experienced senior Python programmer, who can explain complex \
programming features in very understandable way even for beginners. \
Answer normally, no more than in one sentence, but if there is a Python programming related question, laborate your answer in not more than 10 sentences"

## Simple chat with stream

In [84]:
def chat_stream(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = openai.chat.completions.create(model=MODEL_GPT, messages=messages, stream=True)
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

gr.ChatInterface(fn=chat_stream, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7937
* To create a public link, set `share=True` in `launch()`.


## Tools

In [35]:
def chat_with_another_model(message, history):
    history = [] if history.strip() == '' else history
    model='gpt-5-nano-2025-08-07'
    print(f"model switched to {model}")
    print(f"message input to function: {message}")
    print(f"history input to function: {history}")
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=model, messages=messages)
    print(response)
    return response.choices[0].message.content

In [36]:
change_model = {
    "name": "chat_with_another_model",
    "description": "Switch to another GPT model to chat with. Call this whenever user mentions to switch another gpt model",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "message what the user sends",
            },
            "history": {
                "type": "string",
                "description": "chat history",
            },
        },
        "required": ["message", "history"],
        "additionalProperties": False
    }
}

In [37]:
tools = [{"type": "function", "function":change_model}]

In [38]:
def handle_tool_call(message_origin, history_origin):
    # print(f"message_origin:{message_origin}")
    tool_call = message_origin.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    m = arguments.get('message')
    h = arguments.get('history')
    m2 = chat_with_another_model(m, h)
    print(f"m2: {m2}")
    response = {
        "role": "tool",
        "content": m2,
        "tool_call_id": tool_call.id
    }
    return response

In [40]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    print(history)
    response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)
    #print(f"original chat: {response}")

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message, history)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL_GPT, messages=messages)
    
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


[]
[{'role': 'user', 'metadata': None, 'content': 'hello', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': 'Hello! How can I assist you today?', 'options': None}]
[{'role': 'user', 'metadata': None, 'content': 'hello', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': 'Hello! How can I assist you today?', 'options': None}, {'role': 'user', 'metadata': None, 'content': 'how is the day in NY', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': "I don't have real-time data, but you can check a weather website or app for the current conditions in New York.", 'options': None}]
[{'role': 'user', 'metadata': None, 'content': 'hello', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': 'Hello! How can I assist you today?', 'options': None}, {'role': 'user', 'metadata': None, 'content': 'how is the day in NY', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': "I don't have real-time data, but you can c